# TP1 avec SQLite : version courte

ce notebook couvre uniquement les deux premiers fichiers du TP1 : intro Northwind et `SELECT` / `ORDER BY` / `DISTINCT` / `WHERE` / `LIMIT` / `IS NULL`.

## 1. Importer Northwind dans SQLite

Trois méthodes au choix. Toutes finissent par une variable `conn` connectée à la base. Décommenter celle qu'on veut, garder les autres pour référence.

### Méthode 1 : adapter le dump PostgreSQL local

Le fichier `data/northwind.sql` est un export `pg_dump`. Quatre adaptations triviales suffisent pour qu'il passe sous SQLite.

In [ ]:
import os
import re
import sqlite3

DUMP = "../../../data/northwind.sql"
DB = "northwind.sqlite"

with open(DUMP) as f:
    sql = f.read()

sql = re.sub(r"^SET .*?;\s*$", "", sql, flags=re.M)
sql = re.sub(r"ALTER TABLE ONLY[^;]*;", "", sql, flags=re.S)
sql = sql.replace("bytea", "BLOB")
sql = sql.replace(r"'\x'", "NULL")

if os.path.exists(DB):
    os.remove(DB)
conn = sqlite3.connect(DB)
conn.executescript(sql)
conn.commit()

print("OK, customers =", conn.execute("SELECT COUNT(*) FROM customers").fetchone()[0])

### Méthode 1 : ouvrir un fichier `.sqlite` déjà existant

Si quelqu'un a déjà fait la conversion (ou si on a téléchargé un Northwind SQLite), il suffit d'ouvrir le fichier. Zéro réécriture, ouverture instantanée.

In [ ]:
# import sqlite3
# conn = sqlite3.connect("northwind.sqlite")
# print(conn.execute("SELECT COUNT(*) FROM customers").fetchone()[0])

### Méthode 3 : migrer depuis PostgreSQL via pandas

Si PostgreSQL tourne déjà avec la base `northwind`, on copie chaque table en deux lignes : `pd.read_sql_query` côté PG, `df.to_sql` côté SQLite. Pratique pour passer d'un environnement à l'autre sans rejouer le dump.

In [ ]:
# import os, sqlite3, psycopg2, pandas as pd
# from dotenv import load_dotenv
# load_dotenv("../../../.env")
#
# pg = psycopg2.connect(
#     user=os.environ.get("POSTGRESQL_LOCAL_USER"),
#     password=os.environ.get("POSTGRESQL_LOCAL_PASSWORD"),
#     host="localhost", port="5432", dbname="northwind",
# )
# conn = sqlite3.connect("northwind_via_pandas.sqlite")
#
# tables = ["categories", "customers", "employees", "orders",
#           "order_details", "products", "shippers", "suppliers"]
# for t in tables:
#     df = pd.read_sql_query(f"SELECT * FROM {t}", pg)
#     df.to_sql(t, conn, if_exists="replace", index=False)
#     print(t, len(df))
#
# pg.close()

## 2. Helper d'affichage

`q(sql)` renvoie un DataFrame pandas (joli rendu dans Jupyter).

In [ ]:
import pandas as pd

def q(sql):
    return pd.read_sql_query(sql, conn)

## 3. Explorer la base

Sous SQLite, pas de `\dt` ni de `\d` : on utilise `sqlite_master` et `PRAGMA`.

In [ ]:
q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

In [ ]:
q("PRAGMA table_info(orders)")

In [ ]:
q("SELECT * FROM customers LIMIT 5")

## 4. Rapatrier les données

### SELECT

In [ ]:
q("SELECT contact_name, city FROM customers LIMIT 5")

### Alias `AS`

In [ ]:
q("SELECT contact_name || ' ' || contact_title AS whoami FROM customers LIMIT 5")

### ORDER BY

In [ ]:
q("""
SELECT ship_name, order_date
FROM orders
ORDER BY order_date DESC
LIMIT 5
""")

### DISTINCT

In [ ]:
q("SELECT DISTINCT ship_country FROM orders ORDER BY ship_country LIMIT 10")

## 5. Filtrer les données

### WHERE (=, AND, OR, IN, LIKE, BETWEEN, `<>`)

In [ ]:
q("""
SELECT order_id, ship_country, freight
FROM orders
WHERE ship_country IN ('France', 'Germany') AND freight > 50
ORDER BY freight DESC
LIMIT 5
""")

### LIMIT (et OFFSET)

In [ ]:
q("SELECT order_id, order_date FROM orders ORDER BY order_date LIMIT 5 OFFSET 10")

### IS NULL / IS NOT NULL

In [ ]:
q("SELECT order_id, ship_region FROM orders WHERE ship_region IS NULL LIMIT 5")

## Fermeture

In [ ]:
conn.close()